In [1]:
import asyncio
import websockets
import csv
import time
import ssl
import requests
import urllib3
import sys

from websockets.client import connect as ws_connect # Explicit import


# --- CONFIGURATION ---
HOST = "10.37.23.201"
USERNAME = "evo"  # <--- UPDATE THIS
PASSWORD = "root"  # <--- UPDATE THIS

# URL MAPPINGS
BASE_URL = f"http://{HOST}"
# Based on your finding that the endpoint is /status
LOGIN_URL = f"http://{HOST}/login" 
WS_URI = f"wss://{HOST}/"

# FORM FIELDS (Specific to this device)
FIELD_USER = "uname"
FIELD_PASS = "passwd"

# MATH CONSTANTS
SUBPROTOCOL = "ws_ui"
LLMMR = 93206.7555556

# Suppress "Unverified HTTPS request" warnings for clean output
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


async def record_stream(auth_cookie):
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE

    print(f"📡 Connecting to Radar Stream at {WS_URI}...")
    
    headers = {
        "Cookie": auth_cookie,
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Origin": BASE_URL
    }

    try:
        # Use ws_connect() instead of websockets.connect()
        async with ws_connect(
            WS_URI, 
            subprotocols=[SUBPROTOCOL], 
            ssl=ssl_context, 
            extra_headers=headers
        ) as websocket:
            
            print("🔹 Connected! Sending Configuration Request...")
            await websocket.send("GetCfg")
            
            # ... (Rest of your processing logic remains identical) ...
            reference_lat = 0
            reference_lng = 0
            filename = f"radar_data_{int(time.time())}.csv"
            
            with open(filename, 'w', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(['Timestamp', 'ID', 'Class', 'RelX', 'RelY', 'Angle', 'Lat', 'Lng'])
                print(f"🔴 Recording started: {filename}")
                
                async for message in websocket:
                    parts = message.split(";")
                    msg_type = parts[0]
                    
                    if msg_type == "C":
                        try:
                            config_data = parts[1].split(",")
                            reference_lng = float(config_data[12])
                            reference_lat = float(config_data[13])
                            print(f"🗺️  Origin Calibrated: {reference_lat}, {reference_lng}")
                        except: pass
                    
                    elif msg_type == "F":
                        timestamp = time.time()
                        for obj_str in parts[5:]:
                            if not obj_str: continue 
                            obj_data = obj_str.split(",")
                            if len(obj_data) != 5: continue
                            try:
                                o_id = int(obj_data[0])
                                o_cls = int(obj_data[1])
                                o_x = float(obj_data[2])
                                o_y = float(obj_data[3])
                                o_angle = float(obj_data[4])
                                calc_lng = reference_lng + (o_x / LLMMR)
                                calc_lat = reference_lat + (o_y / -LLMMR)
                                writer.writerow([timestamp, o_id, o_cls, o_x, o_y, o_angle, calc_lat, calc_lng])
                                if o_id % 10 == 0: print(f"\r  Tracking Object {o_id}...", end="")
                            except ValueError: continue

    except Exception as e:
        print(f"\n❌ Connection Failed: {e}")


def get_auth_cookie():
    """
    Performs an HTTP POST to the login endpoint to retrieve a session cookie.
    """
    print(f"🔐 Authenticating via POST to {LOGIN_URL}...")
    
    session = requests.Session()
    
    # The data the server expects
    payload = {
        FIELD_USER: USERNAME,
        FIELD_PASS: PASSWORD
    }
    
    try:
        # verify=False ignores the self-signed cert warning
        # allow_redirects=True ensures we catch cookies even if it redirects to a dashboard
        response = session.post(LOGIN_URL, data=payload, verify=False, timeout=10, allow_redirects=True)
        
        # Debugging: Check if login seemed to work
        if response.status_code != 200:
            print(f"⚠️  Server returned status code {response.status_code}")
        
        if not session.cookies:
            print("❌ Login Failed: No cookies received. Check username/password.")
            return None
            
        print("✅ Login Successful. Session Cookie retrieved.")
        
        # Convert the cookie jar into a simple string for the WebSocket
        cookie_string = "; ".join([f"{c.name}={c.value}" for c in session.cookies])
        return cookie_string

    except Exception as e:
        print(f"❌ Authentication Error: {e}")
        return None



C:\Users\rhansen\AppData\Local\Temp\ipykernel_121748\393682019.py:10: DeprecationWarning: websockets.client.connect is deprecated
  from websockets.client import connect as ws_connect # Explicit import


In [2]:
# Run it
cookie = get_auth_cookie()
if cookie:
    await record_stream(cookie)

🔐 Authenticating via POST to http://10.37.23.201/login...
✅ Login Successful. Session Cookie retrieved.
📡 Connecting to Radar Stream at wss://10.37.23.201/...
🔹 Connected! Sending Configuration Request...
🔴 Recording started: radar_data_1765825288.csv
  Tracking Object 488710...🗺️  Origin Calibrated: 44.0846, -116.115
  Tracking Object 488800...

CancelledError: 